In [162]:
import deepchem as dc
from deepchem.models import GCNModel
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from rdkit import Chem
from sklearn.model_selection import train_test_split
# from deepchem.models.lightning.dc_lightning_module import DCLightningModule
# from deepchem.models.lightning.dc_lightning_dataset_module import DCLightningDatasetModule, collate_dataset_wrapper
from deepchem.feat import MolGraphConvFeaturizer
import pytorch_lightning as pl
# from pytorch_lightning.core import LightningModule
from torch.optim import Adam
import torch
import dgl
# from deepchem.feat.graph_data import GraphData
# from torchmetrics import AUROC
# import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchmetrics import AUROC

In [163]:

def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None and mol.GetNumAtoms() > 1
    except Exception:
        return False

def load_single_task_data(target_col):
    """加载单个任务的数据，并仅保留目标值为 0 和 1 的样本"""
    df = pd.read_csv('tox21_cleaned.csv')
    
    df=df[2000:]
    
    
    # 过滤当前任务的缺失值
    df = df.dropna(subset=[target_col])
    print(f"\nTask {target_col} Valid sample size: {len(df)}")
       
 # 过滤有效的 SMILES 并保留索引
    valid_smiles_indices = df['smiles'].apply(is_valid_smiles)
    
    # 过滤目标列值为 0 或 1 的样本
    valid_target_indices = df[target_col].isin([0, 1])
    
    # 合并两个条件，得到最终有效索引
    valid_indices = valid_smiles_indices & valid_target_indices
    valid_df = df[valid_indices]
    
    # 提取有效的 SMILES 和目标值
    valid_smiles = valid_df['smiles'].values
    y = valid_df[target_col].values.astype(np.float32)
    
    return valid_smiles, y

In [164]:
# prepare LightningDataModule
class SmilesDataset(torch.utils.data.Dataset):
    def __init__(self, smiles, labels):
        assert len(smiles) == len(labels)
        featurizer = dc.feat.MolGraphConvFeaturizer()
        X = featurizer.featurize(smiles).flatten()
        self._samples = dc.data.NumpyDataset(X=X, y=labels)
        
    def __len__(self):
        return len(self._samples)
        
    def __getitem__(self, index):
        return (
            self._samples.X[index],
            self._samples.y[index],
            self._samples.w[index],
        )
    
    
class SmilesDatasetBatch:
    def __init__(self, batch):
        X = [np.array([b[0] for b in batch])]
        y = [np.array([b[1] for b in batch])]
        w = [np.array([b[2] for b in batch])]
        self.batch_list = [X, y, w]
        
        
def collate_smiles_dataset_wrapper(batch):
    return SmilesDatasetBatch(batch)

class SmilesDatasetModule(pl.LightningDataModule):
    def __init__(self, train_smiles, train_labels, batch_size):
        super().__init__()
        self._train_smiles = train_smiles
        self._train_labels = train_labels

        self._batch_size = batch_size
        
    def setup(self, stage):
        self.train_dataset = SmilesDataset(
            self._train_smiles,
            self._train_labels,
        )
        

        
    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self._batch_size,
            collate_fn=collate_smiles_dataset_wrapper,
            shuffle=True,  
            drop_last=True,
        )
        
class testSmilesDatasetModule(pl.LightningDataModule):
    def __init__(self, test_smiles, test_labels, batch_size):
        super().__init__()
        self._test_smiles = test_smiles
        self._test_labels = test_labels
        self._batch_size = batch_size

    def setup(self, stage=None):
        self.test_dataset = SmilesDataset(
            self._test_smiles,
            self._test_labels,
        )

    def test_dataloader(self):  # 添加 test_dataloader 方法
        return torch.utils.data.DataLoader(
            self.test_dataset,
            batch_size=self._batch_size,
            collate_fn=collate_smiles_dataset_wrapper,
            shuffle=False,  # 测试集不需要打乱
            drop_last=False,  # 保留最后一个 batch
        )

    def get_test_smiles(self):  # 新增方法，直接返回测试数据的 SMILES
        return self._test_smiles

In [ ]:

class GCNModule(pl.LightningModule):
    def __init__(self, mode, n_tasks, learning_rate):
        super().__init__()
        self.save_hyperparameters(
            "mode",
            "n_tasks",
            "learning_rate",
        )
        self.gcn_model = GCNModel(
            mode=self.hparams.mode,
            n_tasks=self.hparams.n_tasks,
            learning_rate=self.hparams.learning_rate,
        )
        self.pt_model = self.gcn_model.model
        self.loss = self.gcn_model._loss_fn
        
    def configure_optimizers(self):
        return self.gcn_model.optimizer._create_pytorch_optimizer(
            self.pt_model.parameters(),
        )
    
    def training_step(self, batch, batch_idx):
        batch = batch.batch_list
        inputs, labels, weights = self.gcn_model._prepare_batch(batch)
        outputs = self.pt_model(inputs)
        
        if isinstance(outputs, torch.Tensor):
            outputs = [outputs]
    
        if self.gcn_model._loss_outputs is not None:
            outputs = [outputs[i] for i in self.gcn_model._loss_outputs]
    

            
        loss_outputs = self.loss(outputs, labels, weights)
        
        self.log(
            "train_loss",
            loss_outputs,
            on_epoch=True,
            sync_dist=True,
            reduce_fx="mean",
            prog_bar=True,
            batch_size=32
        )
        
        return loss_outputs
    
        # 在 test_step 里计算 AUC
    def test_step(self, batch, batch_idx):
        batch = batch.batch_list
        inputs, labels, weights= self.gcn_model._prepare_batch(batch)
        smiles = self.trainer.datamodule.get_test_smiles()
        print(f"smiles: {smiles}")
        featurizer = MolGraphConvFeaturizer()
        X =featurizer.featurize(smiles)
        
        
        # 将测试数据封装为 DeepChem 的 Dataset 对象
        test_dataset = dc.data.NumpyDataset(X=X, y=labels)
        # 评估指标
        metrics = [
        dc.metrics.Metric(dc.metrics.roc_auc_score, name="AUC"),
        dc.metrics.Metric(dc.metrics.f1_score, name="F1"),
        dc.metrics.Metric(dc.metrics.recall_score, name="Recall"),
        dc.metrics.Metric(dc.metrics.accuracy_score, name="Accuracy")
        ]

        # 生成DataFrame
        results = self.gcn_model.evaluate(test_dataset, metrics)
        performance_df = pd.DataFrame([results], columns=results.keys())
        # # 取 sigmoid 输出，用于二分类 AUC
        # probs = torch.sigmoid(outputs) if isinstance(outputs, torch.Tensor) else torch.sigmoid(outputs[0])
        # self.test_auc_metric.update(probs.view(-1), labels.view(-1).long())
        # # 随便返回个数字做 test_loss
        return performance_df



In [166]:
# # create module objects
# smiles_datasetmodule = SmilesDatasetModule(
#     train_smiles=["C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC"],
#     train_labels=[0., 1., 0., 1., 0., 1., 0., 1., 0., 1.],
#     batch_size=10,
# )

# gcnmodule = GCNModule(
#     mode="classification",
#     n_tasks=1,
#     learning_rate=1e-3,
# )

# trainer = pl.Trainer(
#     max_epochs=5,
# )

# # train
# trainer.fit(
#     model=gcnmodule,
#     datamodule=smiles_datasetmodule,
# )

In [167]:
def train_single_task(target_col):
    smiles, y = load_single_task_data(target_col)
    X_train, X_test, y_train, y_test = train_test_split(
        smiles, y, test_size=0.2, random_state=42
    )
    # # 验证集
    # X_val, X_test, y_val, y_test = train_test_split(
    #     X_test, y_test, test_size=0.5, random_state=42
    # )

    # 训练数据模块
    smiles_datasetmodule = SmilesDatasetModule(
        train_smiles=X_train,
        train_labels=y_train,

        batch_size=32,
    )
    gcnmodule = GCNModule(
        mode='classification',
        n_tasks=1,
        learning_rate=0.001
    )

    trainer = pl.Trainer(
        max_epochs=1,
        accelerator='cpu',
    )

    # 训练
    trainer.fit(
        model=gcnmodule,
        datamodule=smiles_datasetmodule,
    )

    # 测试数据模块
    smiles_datasetmodule = testSmilesDatasetModule(

        test_smiles=X_test,   # 测试集单独传
        test_labels=y_test,
        batch_size=32,
    )
    

    # 测试并打印结果
    test_result = trainer.test(model=gcnmodule, datamodule=smiles_datasetmodule)
    # print("Test AUC:", test_result[0].get("test_auc", None))
    # return test_result[0].get("test_auc", None)
    print("Test Result:", test_result)
    return test_result



In [168]:

TARGET_COLS = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 
              'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE',
              'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

if __name__ == "__main__":
    result = {}
    for target in TARGET_COLS[:1]:
        performance = train_single_task(target)
        result[target] = performance 



Task NR-AR Valid sample size: 5567


You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type | Params | Mode 
------------------------------------------
0 | pt_model | GCN  | 29.4 K | train
------------------------------------------
29.4 K    Trainable params
0         Non-trainable params
29.4 K    Total params
0.118     Total estimated model params size (MB)
24        Modules in train mode
0         Modules in eval mode
d:\Anaconda\envs\chem311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 138/138 [00:08<00:00, 16.69it/s, v_num=95, train_loss_step=0.195, train_loss_epoch=0.389]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 138/138 [00:08<00:00, 16.61it/s, v_num=95, train_loss_step=0.195, train_loss_epoch=0.389]


d:\Anaconda\envs\chem311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0:   0%|          | 0/35 [00:00<?, ?it/s]smiles: ['Cc1cccc2sc3nncn3c12' 'CC(C)(Oc1ccc(Cl)cc1)C(=O)O'
 'CCC(=O)OC/C=C/c1ccccc1' ... 'COc1ccc(C)cc1'
 'Cc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1' 'CCC(C)N']
X: [GraphData(node_features=[13, 30], edge_index=[2, 30], edge_features=None)
 GraphData(node_features=[14, 30], edge_index=[2, 28], edge_features=None)
 GraphData(node_features=[14, 30], edge_index=[2, 28], edge_features=None)
 ...
 GraphData(node_features=[9, 30], edge_index=[2, 18], edge_features=None)
 GraphData(node_features=[24, 30], edge_index=[2, 52], edge_features=None)
 GraphData(node_features=[5, 30], edge_index=[2, 8], edge_features=None)]
labels: [tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.])]


AttributeError: 'list' object has no attribute 'shape'

In [ ]:
print(result)

{}


In [ ]:
# smiles = ["C1CCC1", "CCC"]
# labels = [0., 1.]
# featurizer = dc.feat.MolGraphConvFeaturizer()
# X = featurizer.featurize(smiles)
# dataset = dc.data.NumpyDataset(X=X, y=labels)

# model = GCNModel(
#     mode='classification',
#     n_tasks=1,
#     batch_size=2,
#     learning_rate=0.001
# )

# loss = model.fit(dataset, nb_epoch=5)
# print(loss)

In [ ]:


# def calculate_test_auc(model, test_smiles, test_labels):
#     """直接计算测试集AUC的完整流程"""
    
#     # 1. 准备测试数据集
#     test_dataset = SmilesDataset(test_smiles, test_labels)
#     test_loader = DataLoader(
#         test_dataset,
#         batch_size=32,
#         collate_fn=collate_smiles_dataset_wrapper,
#         shuffle=False
#     )
    
#     # 2. 模型设为评估模式
#     model.eval()
#     all_probs = []
#     all_labels = []
    
#     # 3. 进行预测
#     with torch.no_grad():
#         for batch in test_loader:
#             # 前向传播
#             outputs = model.pt_model(batch.X)
            
#             # 计算概率
#             if model.hparams.mode == "classification":
#                 if model.hparams.n_tasks == 1:
#                     probs = torch.sigmoid(outputs).cpu().numpy()  # 二分类
#                 else:
#                     probs = torch.softmax(outputs, dim=1).cpu().numpy()  # 多分类
#             else:
#                 raise ValueError("AUC只适用于分类任务")
            
#             # 收集结果
#             all_probs.append(probs)
#             all_labels.append(batch.y.cpu().numpy())
    
#     # 4. 合并结果
#     probs = np.concatenate(all_probs, axis=0)
#     labels = np.concatenate(all_labels, axis=0)
    
#     # 5. 计算AUC
#     if model.hparams.mode == "classification":
#         if model.hparams.n_tasks == 1:  # 二分类
#             auc = roc_auc_score(labels, probs)
#         else:  # 多分类
#             auc = roc_auc_score(labels, probs, multi_class='ovo')
            
#         print(f"Test AUC: {auc:.4f}")
#         return auc
#     else:
#         print("回归任务无法计算AUC")
#         return None

# # 使用示例 --------------------------------------------------
# # 假设已有训练好的模型和测试数据
# # trained_model = GCNModule.load_from_checkpoint("best_model.ckpt")  # 加载训练好的模型
# trained_model = gcnmodule
# test_smiles = ["CCCC", "CN", "C1CCCC1", "CCO"]  # 测试集SMILES
# test_labels = [0, 1, 0, 1]  # 测试标签

# # 直接计算AUC
# auc_score = calculate_test_auc(
#     trained_model, 
#     test_smiles, 
#     test_labels
# )

# # 可视化ROC曲线（可选）
# import matplotlib.pyplot as plt
# from sklearn.metrics import RocCurveDisplay

# if trained_model.hparams.mode == "classification":
#     if trained_model.hparams.n_tasks == 1:
#         RocCurveDisplay.from_predictions(
#             test_labels,
#             probs,
#             name="ROC Curve",
#             plot_chance_level=True
#         )
#     else:
#         for class_id in range(trained_model.hparams.n_tasks):
#             RocCurveDisplay.from_predictions(
#                 (labels == class_id).astype(int),
#                 probs[:, class_id],
#                 name=f"Class {class_id} vs Rest"
#             )
#     plt.show()

AttributeError: 'SmilesDatasetBatch' object has no attribute 'X'

In [ ]:
#测试
# create module objects
smiles_datasetmodule = SmilesDatasetModule(
    train_smiles=["C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC"],
    train_labels=[0., 1., 0., 1., 0., 1., 0., 1., 0., 1.],
    batch_size=10,
)

valid_dataset = SmilesDatasetModule(
    train_smiles=["C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC", "C1CCC1", "CCC"],
    train_labels=[0., 1., 0., 1., 0., 1., 0., 1., 0., 1.],
    batch_size=10,
)


testgcnmodule = GCNModule(
    mode="classification",
    n_tasks=1,
    learning_rate=1e-3,
)

testtrainer = pl.Trainer(
    max_epochs=5,
)

# train
testtrainer.fit(
    model=testgcnmodule,
    datamodule=smiles_datasetmodule,
)



You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type | Params | Mode 
------------------------------------------
0 | pt_model | GCN  | 29.4 K | train
------------------------------------------
29.4 K    Trainable params
0         Non-trainable params
29.4 K    Total params
0.118     Total estimated model params size (MB)
24        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 1/1 [00:00<00:00, 18.85it/s, v_num=70, train_loss_step=0.108, train_loss_epoch=0.108]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 1/1 [00:00<00:00, 11.50it/s, v_num=70, train_loss_step=0.108, train_loss_epoch=0.108]
